# 03 · Statistical models: exponential smoothing, ARIMA, Theta

These models assume a parametric *data-generating process*, fit few parameters by maximum likelihood, and produce **analytic prediction intervals**. They shine on short, single series like the airline data.

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
from tsforecasting.data.data_loader import load_airline, load_jena, temporal_split
from tsforecasting.features.build_features import TARGET

try:
    df = load_jena()
except FileNotFoundError:  # first run: download + clean + features
    from tsforecasting.data import make_dataset
    from tsforecasting.features import build_features
    make_dataset.main(); build_features.main()
    df = load_jena()
y = df[TARGET]
airline = load_airline()
df.shape, airline.shape

In [ ]:
from tsforecasting.evaluation import backtest, score
from tsforecasting.visualization.visualize import plot_forecast

train, test = airline.iloc[:-24], airline.iloc[-24:]
H = len(test)

## 1. Exponential smoothing (ETS)

**Simple exponential smoothing** forecasts with a weighted average whose weights decay geometrically:

$$\hat y_{T+1} = \alpha y_T + (1-\alpha)\hat y_T = \sum_{j\ge0}\alpha(1-\alpha)^j y_{T-j}.$$

**Holt-Winters** adds trend $b_t$ and seasonal $s_t$ states (additive form):

$$
\begin{aligned}
\ell_t &= \alpha(y_t - s_{t-m}) + (1-\alpha)(\ell_{t-1}+b_{t-1})\\
b_t &= \beta(\ell_t-\ell_{t-1}) + (1-\beta)b_{t-1}\\
s_t &= \gamma(y_t-\ell_{t-1}-b_{t-1}) + (1-\gamma)s_{t-m}\\
\hat y_{T+h} &= \ell_T + h\,b_T + s_{T+h-m\lceil h/m\rceil}
\end{aligned}
$$

The ETS framework (Error, Trend, Seasonal) writes these as state-space models, which yields a likelihood and prediction intervals.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

fits = {
    "additive seasonality": ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12).fit(),
    "multiplicative seasonality": ExponentialSmoothing(train, trend="add", seasonal="mul", seasonal_periods=12).fit(),
}
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, (name, f) in zip(axes, fits.items()):
    plot_forecast(train, pd.DataFrame({"mean": f.forecast(H)}), test, title=f"Holt-Winters, {name}", ax=ax)
fig.tight_layout()
pd.DataFrame({k: v.params for k, v in fits.items()}).loc[
    ["smoothing_level", "smoothing_trend", "smoothing_seasonal"]].round(3)

The multiplicative model follows the growing seasonal swings. The additive one underestimates the peaks. This matches what we saw in notebook 01.

### AutoETS (sktime)
`AutoETS(auto=True)` searches the combinations of error/trend/seasonal components and picks the one with the lowest AIC $= 2k - 2\ln\hat L$.

In [ ]:
from sktime.forecasting.ets import AutoETS

auto_ets = AutoETS(auto=True, sp=12, n_jobs=-1).fit(train)
print(auto_ets.summary().tables[0])
fc = auto_ets.predict(fh=list(range(1, H + 1)))
# ponytail: point forecast only; sktime 1.2 AutoETS.predict_interval breaks on current statsmodels (simulate_kwargs)
plot_forecast(train, pd.DataFrame({"mean": fc.to_numpy()}), test, title="AutoETS (selected by AIC)");

## 2. ARIMA and SARIMA

With the backshift operator $B y_t = y_{t-1}$, an ARIMA$(p,d,q)$ model is

$$\underbrace{\phi(B)}_{\text{AR}}\ (1-B)^d\, y_t = \underbrace{\theta(B)}_{\text{MA}}\,\varepsilon_t,\qquad \varepsilon_t\sim\mathcal N(0,\sigma^2),$$

with $\phi(B)=1-\phi_1B-\dots-\phi_pB^p$ and $\theta(B)=1+\theta_1B+\dots+\theta_qB^q$. The seasonal version SARIMA$(p,d,q)(P,D,Q)_m$ multiplies by seasonal polynomials in $B^m$:

$$\phi(B)\,\Phi(B^m)\,(1-B)^d(1-B^m)^D\,y_t = \theta(B)\,\Theta(B^m)\,\varepsilon_t.$$

Box & Jenkins' **airline model** is SARIMA$(0,1,1)(0,1,1)_{12}$ on $\log y_t$. It matches the identification of notebook 01: one regular and one seasonal difference, with MA spikes at lags 1 and 12.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima = SARIMAX(np.log(train), order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
print(sarima.summary().tables[1])
pred = sarima.get_forecast(H)
ci = np.exp(pred.conf_int(alpha=0.2))  # back-transform: intervals stay valid, the mean becomes the median
plot_forecast(train, pd.DataFrame({"mean": np.exp(pred.predicted_mean).to_numpy(),
                                   "q10": ci.iloc[:, 0].to_numpy(), "q90": ci.iloc[:, 1].to_numpy()}),
              test, title="Airline model SARIMA(0,1,1)(0,1,1)$_{12}$ on log scale");

### Residual diagnostics

If the model captured all the structure, the residuals $\hat\varepsilon_t$ should be **white noise**: no autocorrelation, ideally Gaussian. The **Ljung-Box** statistic

$$Q = T(T+2)\sum_{k=1}^{K}\frac{\hat\rho_k^2}{T-k}\ \sim\ \chi^2_{K-p-q}$$

tests $H_0$: no autocorrelation up to lag $K$. A large p-value is good.

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

sarima.plot_diagnostics(figsize=(12, 7)); plt.tight_layout()
acorr_ljungbox(sarima.resid.iloc[13:], lags=[12, 24]).round(3)

## 3. The Theta method

The Theta method (Assimakopoulos & Nikolopoulos, 2000) won the M3 competition. It works as follows:
1. Deseasonalize.
2. Combine a linear-trend line ($\theta=0$) with simple exponential smoothing of a curvature-amplified series ($\theta=2$).
3. Reseasonalize.

Hyndman & Billah (2003) showed it is equivalent to SES with drift equal to half the trend slope.

In [ ]:
from sktime.forecasting.theta import ThetaForecaster

theta = ThetaForecaster(sp=12).fit(train)
iv = theta.predict_interval(fh=list(range(1, H + 1)), coverage=0.8)
plot_forecast(train, pd.DataFrame({"mean": theta.predict(fh=list(range(1, H + 1))).to_numpy(),
                                   "q10": iv.iloc[:, 0].to_numpy(), "q90": iv.iloc[:, 1].to_numpy()}),
              test, title="Theta (multiplicative deseasonalization)");

## 4. Fair comparison: rolling-origin backtest on airline

We wrap each model as a `forecast_fn` and run the protocol from notebook 02 (initial = 96 months, $H$ = 12, step = 6).

In [ ]:
from tsforecasting.models.baselines import seasonal_naive
from tsforecasting.evaluation import QUANTILES, qcol
from scipy.stats import norm

def sarima_fn(y, h, *_):
    r = SARIMAX(np.log(y), order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False).get_forecast(h)
    mu, sd = r.predicted_mean.to_numpy(), np.sqrt(r.var_pred_mean.to_numpy())
    out = pd.DataFrame({qcol(q): np.exp(mu + norm.ppf(q) * sd) for q in QUANTILES})
    return out.assign(mean=np.exp(mu))

def hw_fn(y, h, *_):
    f = ExponentialSmoothing(y, trend="add", seasonal="mul", seasonal_periods=12).fit()
    return pd.DataFrame({"mean": f.forecast(h).to_numpy()})

def theta_fn(y, h, *_):
    return pd.DataFrame({"mean": ThetaForecaster(sp=12).fit(y).predict(fh=list(range(1, h + 1))).to_numpy()})

fns = {"seasonal_naive": seasonal_naive(12), "holt_winters_mul": hw_fn, "theta": theta_fn, "sarima_airline": sarima_fn}
res = {k: backtest(f, airline, initial=96, horizon=12, step=6, m=12) for k, f in fns.items()}
board = pd.DataFrame({k: score(r) for k, r in res.items()}).T.sort_values("mase")
board.round(3)

In [ ]:
from tsforecasting.visualization.visualize import plot_backtest, plot_leaderboard

fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [1, 2]})
plot_leaderboard(board, ax=axes[0]).set_title("MASE")
plot_backtest(res["sarima_airline"], airline, title="SARIMA folds with 80% band", ax=axes[1])
fig.tight_layout()

## 5. Limits on high-frequency data

On hourly Jena temperature, these models must handle $m=24$ *and* a yearly cycle of $m\approx 8766$. Holt-Winters only takes a single seasonal period, and SARIMA with $m=8766$ is computationally infeasible. In `make train` they are refit on the last 4 weeks only and land **behind seasonal naive**. This motivates the ML and foundation models next.

In [ ]:
from tsforecasting.models.train_model import ets_fn, theta_fn as jena_theta_fn

window = y.iloc[-24 * 35 : -24 * 7]
actual = y.iloc[-24 * 7 : -24 * 6]
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5), sharey=True)
for ax, (name, fn) in zip(axes, {"Holt-Winters (damped)": ets_fn, "Theta": jena_theta_fn}.items()):
    plot_forecast(window.iloc[-24 * 5 :], fn(window, 24).set_index(actual.index), actual, title=name, ax=ax)
fig.tight_layout()

## Takeaways
- ETS and (S)ARIMA are interpretable and come with likelihood-based intervals. On the airline series the 40-year-old airline model is hard to beat.
- Residual diagnostics tell you *what structure is left*. That makes them the statistician's debugger.
- Long, multi-seasonal, multivariate series push these models beyond their comfort zone.